In [ ]:
# Task1: Symbolic, unconditioned generation
Using the LSTM framework

In [38]:
"""
Central configuration for Task 1
"""

import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import argparse
import math
import time
import json
import mido


# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR = os.getcwd()
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
OUTPUT_DIR    = os.path.join(BASE_DIR, "outputs")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Vocabulary  ────────────────────────────────────────────────────────────────
VOCAB_SIZE        = 389
PAD_TOKEN         = 388
NOTE_ON_OFFSET    = 0
NOTE_OFF_OFFSET   = 128
TIME_SHIFT_OFFSET = 256
VELOCITY_OFFSET   = 356
N_TIME_STEPS      = 100
N_VELOCITY_BINS   = 32
TIME_STEP_MS      = 10
VELOCITY_BIN_SIZE = 4

# ── Data ───────────────────────────────────────────────────────────────────────
SEQ_LEN    = 512      # window size from preprocess.py
INPUT_LEN  = SEQ_LEN - 1   # 511: input  = window[:-1]
TARGET_LEN = SEQ_LEN - 1   # 511: target = window[1:]

# ── Model architecture ─────────────────────────────────────────────────────────
EMBED_DIM    = 256
HIDDEN_SIZE  = 512
NUM_LAYERS   = 2
DROPOUT      = 0.3

# ── Training ───────────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LEARNING_RATE = 1e-3
LR_PATIENCE  = 3        # ReduceLROnPlateau patience (epochs)
LR_FACTOR    = 0.5      # LR multiplier on plateau
GRAD_CLIP    = 1.0      # gradient clipping max norm
EARLY_STOP_PATIENCE = 6 # stop if val loss doesn't improve

# ── Generation ─────────────────────────────────────────────────────────────────
GENERATE_TOKENS  = 1024  # number of new tokens to generate
TEMPERATURE      = 1.0   # sampling temperature (higher = more random)
TOP_K            = 40    # top-k sampling
SEED_LENGTH      = 64    # seed tokens taken from a real piece

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42


In [11]:
"""
Dataset and DataLoader utilities for Task 1 training.
Loads preprocessed *_task1.npy windows produced by preprocess.py.
"""

class MusicWindowDataset(Dataset):
    """
    Each item is one sliding window of event tokens from preprocess.py.
    Returns (input_ids, target_ids), both length 511:
        input_ids  = window[:-1]   (tokens 0..510)
        target_ids = window[1:]    (tokens 1..511)
    PAD tokens (388) at the end of the last window are handled by
    CrossEntropyLoss(ignore_index=PAD_TOKEN) during training.
    """

    def __init__(self, split: str, processed_dir: str = PROCESSED_DIR):
        path = os.path.join(processed_dir, f"{split}_task1.npy")
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Preprocessed data not found: {path}\n"
                f"Run `python preprocess.py` first."
            )
        # Load as int32, convert to long for embedding lookup
        data = np.load(path)                        # (N, 512)
        self.data = torch.tensor(data, dtype=torch.long)
        print(f"[Dataset] {split}: {len(self.data)} windows  "
              f"shape={tuple(self.data.shape)}")

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        window = self.data[idx]                     # (512,)
        return window[:-1], window[1:]              # (511,), (511,)


def get_dataloaders(processed_dir: str = PROCESSED_DIR,
                    batch_size: int = BATCH_SIZE,
                    num_workers: int = 2):
    """
    Build train / validation / test DataLoaders.
    Returns a dict: {'train': ..., 'validation': ..., 'test': ...}
    """
    g = torch.Generator()
    g.manual_seed(SEED)

    loaders = {}
    for split in ("train", "validation", "test"):
        dataset = MusicWindowDataset(split, processed_dir)
        shuffle = (split == "train")
        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=True,
            generator=g if shuffle else None,
        )
    return loaders


loaders = get_dataloaders()
inp, tgt = next(iter(loaders["train"]))
print(f"input shape : {inp.shape}")    # (B, 511)
print(f"target shape: {tgt.shape}")   # (B, 511)
print(f"token range : {inp.min().item()} – {inp.max().item()}")
pad_frac = (tgt == PAD_TOKEN).float().mean().item()
print(f"PAD fraction: {pad_frac:.3%}")


[Dataset] train: 119817 windows  shape=(119817, 512)
[Dataset] validation: 13538 windows  shape=(13538, 512)
[Dataset] test: 15613 windows  shape=(15613, 512)
input shape : torch.Size([32, 511])
target shape: torch.Size([32, 511])
token range : 21 – 385
PAD fraction: 0.000%


In [16]:
"""
Architecture:
    Embedding(389, 256)
    → Dropout
    → LSTM(256 → 512, 2 layers, dropout=0.3)
    → Dropout
    → Linear(512 → 389)

Training:  next-token prediction with CrossEntropyLoss (ignore PAD=388)
Inference: autoregressive sampling with temperature + top-k
"""

class MusicLSTM(nn.Module):

    def __init__(
        self,
        vocab_size:  int = VOCAB_SIZE,
        embed_dim:   int = EMBED_DIM,
        hidden_size: int = HIDDEN_SIZE,
        num_layers:  int = NUM_LAYERS,
        dropout:     float = DROPOUT,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # Embedding: PAD token gets a zero vector and is not updated
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=PAD_TOKEN
        )

        # LSTM: dropout only applied between layers (not after last layer)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)

        # Project LSTM output to vocabulary logits
        self.fc = nn.Linear(hidden_size, vocab_size)

        self._init_weights()

    def _init_weights(self):
        """Xavier uniform for embedding + linear; orthogonal for LSTM weights."""
        nn.init.xavier_uniform_(self.embedding.weight)
        # Zero out the PAD embedding row
        with torch.no_grad():
            self.embedding.weight[PAD_TOKEN].fill_(0)

        for name, param in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    # ── Forward pass ──────────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor, hidden=None):
        """
        Args:
            x      : (B, T) long tensor of token IDs
            hidden : tuple (h, c) each (num_layers, B, hidden_size), or None

        Returns:
            logits : (B, T, vocab_size)
            hidden : tuple (h, c) — detached for next chunk if needed
        """
        emb = self.dropout(self.embedding(x))       # (B, T, embed_dim)
        out, hidden = self.lstm(emb, hidden)         # (B, T, hidden_size)
        logits = self.fc(self.dropout(out))          # (B, T, vocab_size)
        return logits, hidden

    # ── Hidden state helpers ──────────────────────────────────────────────────

    def init_hidden(self, batch_size: int, device: torch.device):
        """Return zero-initialised (h_0, c_0)."""
        h = torch.zeros(self.num_layers, batch_size, self.hidden_size,
                        device=device)
        c = torch.zeros_like(h)
        return (h, c)

    @staticmethod
    def detach_hidden(hidden):
        """Detach hidden state from computation graph (truncated BPTT)."""
        h, c = hidden
        return h.detach(), c.detach()

    # ── Generation ────────────────────────────────────────────────────────────

    @torch.no_grad()
    def generate(
        self,
        seed_tokens:      list,
        max_new_tokens:   int   = 1024,
        temperature:      float = 1.0,
        top_k:            int   = 40,
        device:           torch.device = torch.device("cpu"),
    ) -> list:
        """
        Autoregressive generation with top-k sampling.

        Args:
            seed_tokens    : list of int, warm-up context (e.g. 64 tokens)
            max_new_tokens : how many new tokens to generate
            temperature    : >1 more random, <1 more deterministic
            top_k          : keep only top-k logits before sampling

        Returns:
            List[int] — seed_tokens + generated tokens
        """
        self.eval()

        # Encode seed to build hidden state
        seed = torch.tensor(seed_tokens, dtype=torch.long,
                            device=device).unsqueeze(0)  # (1, seed_len)
        logits, hidden = self.forward(seed)
        next_logits = logits[0, -1, :]                   # (vocab_size,)

        generated = list(seed_tokens)

        for _ in range(max_new_tokens):
            # Apply temperature
            scaled = next_logits / max(temperature, 1e-8)

            # Top-k filtering
            top_vals, top_idx = torch.topk(scaled, min(top_k, scaled.size(-1)))
            probs = torch.softmax(top_vals, dim=-1)
            chosen_pos = torch.multinomial(probs, num_samples=1)
            token = top_idx[chosen_pos].item()

            generated.append(token)

            # Single-step forward (reuse hidden state — O(1) per step)
            x_t = torch.tensor([[token]], dtype=torch.long, device=device)
            logits_t, hidden = self.forward(x_t, hidden)
            next_logits = logits_t[0, 0, :]

        return generated

    # ── Utility ───────────────────────────────────────────────────────────────

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def __repr__(self):
        return (
            f"MusicLSTM("
            f"vocab={VOCAB_SIZE}, embed={EMBED_DIM}, "
            f"hidden={self.hidden_size}, layers={self.num_layers}, "
            f"params={self.count_parameters():,})"
        )



model = MusicLSTM()
print(model)

B, T = 4, 511
x = torch.randint(0, VOCAB_SIZE - 1, (B, T))   # exclude PAD
logits, hidden = model(x)
print(f"logits shape : {logits.shape}")         # (4, 511, 389)
print(f"hidden h     : {hidden[0].shape}")      # (2, 4, 512)


MusicLSTM(vocab=389, embed=256, hidden=512, layers=2, params=3,977,349)
logits shape : torch.Size([4, 511, 389])
hidden h     : torch.Size([2, 4, 512])


In [18]:
"""
Training pipeline
"""

# ── Reproducibility ────────────────────────────────────────────────────────────

def set_seed(seed: int):
    import random, numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ── One epoch ─────────────────────────────────────────────────────────────────

def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train(train)
    total_loss = 0.0
    total_tokens = 0

    for input_ids, target_ids in loader:
        input_ids  = input_ids.to(device)   # (B, 511)
        target_ids = target_ids.to(device)  # (B, 511)

        logits, _ = model(input_ids)        # (B, 511, vocab_size)

        # Flatten for CrossEntropyLoss
        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),  # (B*511, vocab_size)
            target_ids.reshape(-1)           # (B*511,)
        )

        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        # Count non-PAD tokens for accurate loss tracking
        non_pad = (target_ids != PAD_TOKEN).sum().item()
        total_loss   += loss.item() * non_pad
        total_tokens += non_pad

    avg_loss = total_loss / max(total_tokens, 1)
    perplexity = math.exp(min(avg_loss, 20))     # cap to avoid overflow
    return avg_loss, perplexity


# ── Checkpoint helpers ─────────────────────────────────────────────────────────

def save_checkpoint(model, optimizer, epoch, val_loss, path):
    torch.save({
        "epoch":      epoch,
        "val_loss":   val_loss,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
    }, path)


def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"])
    if optimizer and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt["epoch"], ckpt["val_loss"]


# ── Main training loop ─────────────────────────────────────────────────────────

def train(args):
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Data
    loaders = get_dataloaders(batch_size=args.batch_size)

    # Model
    model = MusicLSTM().to(device)
    print(model)

    # Loss, optimiser, scheduler
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=LR_PATIENCE, factor=LR_FACTOR,
        verbose=True
    )

    # Training state
    best_val_loss    = float("inf")
    no_improve_count = 0
    history = {"train_loss": [], "val_loss": [],
               "train_ppl":  [], "val_ppl":  []}

    print(f"\n{'Epoch':>5} {'Train Loss':>11} {'Train PPL':>10} "
          f"{'Val Loss':>10} {'Val PPL':>9} {'Time':>7}")
    print("-" * 58)

    for epoch in range(1, args.epochs + 1):
        t0 = time.time()

        train_loss, train_ppl = run_epoch(
            model, loaders["train"], criterion, optimizer, device, train=True
        )
        with torch.no_grad():
            val_loss, val_ppl = run_epoch(
                model, loaders["validation"], criterion, None, device, train=False
            )

        elapsed = time.time() - t0
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_ppl"].append(train_ppl)
        history["val_ppl"].append(val_ppl)

        print(f"{epoch:>5} {train_loss:>11.4f} {train_ppl:>10.2f} "
              f"{val_loss:>10.4f} {val_ppl:>9.2f} {elapsed:>6.0f}s")

        # Save best checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve_count = 0
            ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
            save_checkpoint(model, optimizer, epoch, val_loss, ckpt_path)
            print(f"  ✓ Saved best model (val_loss={val_loss:.4f})")
        else:
            no_improve_count += 1
            if no_improve_count >= EARLY_STOP_PATIENCE:
                print(f"\nEarly stopping after {epoch} epochs "
                      f"(no improvement for {EARLY_STOP_PATIENCE} epochs).")
                break

        # Also save latest checkpoint (useful for resuming)
        save_checkpoint(
            model, optimizer, epoch, val_loss,
            os.path.join(CHECKPOINT_DIR, "latest_model.pt")
        )

    # Final test evaluation using best weights
    print("\nLoading best model for test evaluation…")
    load_checkpoint(os.path.join(CHECKPOINT_DIR, "best_model.pt"), model)
    with torch.no_grad():
        test_loss, test_ppl = run_epoch(
            model, loaders["test"], criterion, None, device, train=False
        )
    print(f"Test Loss: {test_loss:.4f}   Test Perplexity: {test_ppl:.2f}")

    # Save training history
    import json
    hist_path = os.path.join(CHECKPOINT_DIR, "history.json")
    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2)
    print(f"History saved to {hist_path}")

    return history


# ── CLI ───────────────────────────────────────────────────────────────────────

epochs = NUM_EPOCHS
batch_size = BATCH_SIZE
lr = LEARNING_RATE

class Args:
    pass

args = Args()
args.epochs = epochs
args.batch_size = batch_size
args.lr = lr

train(args)


Device: cuda
[Dataset] train: 119817 windows  shape=(119817, 512)
[Dataset] validation: 13538 windows  shape=(13538, 512)
[Dataset] test: 15613 windows  shape=(15613, 512)
MusicLSTM(vocab=389, embed=256, hidden=512, layers=2, params=3,977,349)

Epoch  Train Loss  Train PPL   Val Loss   Val PPL    Time
----------------------------------------------------------


/opt/conda/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


    1      2.5855      13.27     2.2465      9.45    478s
  ✓ Saved best model (val_loss=2.2465)
    2      2.1806       8.85     2.1686      8.75    480s
  ✓ Saved best model (val_loss=2.1686)
    3      2.1071       8.22     2.1373      8.48    480s
  ✓ Saved best model (val_loss=2.1373)
    4      2.0680       7.91     2.1224      8.35    480s
  ✓ Saved best model (val_loss=2.1224)
    5      2.0425       7.71     2.1098      8.25    481s
  ✓ Saved best model (val_loss=2.1098)
    6      2.0240       7.57     2.1057      8.21    481s
  ✓ Saved best model (val_loss=2.1057)
    7      2.0098       7.46     2.1010      8.17    481s
  ✓ Saved best model (val_loss=2.1010)
    8      1.9988       7.38     2.0971      8.14    481s
  ✓ Saved best model (val_loss=2.0971)
    9      1.9895       7.31     2.0952      8.13    481s
  ✓ Saved best model (val_loss=2.0952)
   10      1.9817       7.25     2.0910      8.09    482s
  ✓ Saved best model (val_loss=2.0910)
   11      1.9749       7.21  

{'train_loss': [2.58549265552491,
  2.1806264685714436,
  2.107092329186182,
  2.0680339072510843,
  2.0425054080155673,
  2.024007842251443,
  2.0098302749551986,
  1.9988271668381459,
  1.9894796582149183,
  1.9816537364510811,
  1.9748541209772763,
  1.968777577472246,
  1.9637226163916228,
  1.9591177595127047,
  1.9547888531863646,
  1.950892896627648,
  1.9474628758007597,
  1.9442241925526864,
  1.9409122520959792,
  1.9387037057099727,
  1.936162429608497,
  1.933512289482697,
  1.9310363354123146,
  1.9288218915525144,
  1.9266879826875174,
  1.9245990333276786,
  1.9229159415220438,
  1.9210662850402276,
  1.919441223366952,
  1.9176989617876097],
 'val_loss': [2.2464621531616173,
  2.168556980223302,
  2.137292420664689,
  2.1224135456526514,
  2.109765263596404,
  2.105739336616014,
  2.101029625676076,
  2.0970753022162567,
  2.0952440052281367,
  2.09095762024012,
  2.0907990199713344,
  2.0914852427251307,
  2.0886426281055672,
  2.0926427047906455,
  2.0895965908790717,

In [51]:
"""
Music generation script for Task 1

Loads best_model.pt, samples a seed from the test set,
runs autoregressive generation, and writes a MIDI file.
"""
# ── Token diagnostics ─────────────────────────────────────────────────────────

def diagnose_tokens(token_seq: list):
    """Print token type breakdown and estimated total duration."""
    counts = {"NOTE_ON": 0, "NOTE_OFF": 0, "TIME_SHIFT": 0, "VELOCITY": 0, "PAD": 0}
    total_ms = 0
    for t in token_seq:
        if t < 128:                   counts["NOTE_ON"] += 1
        elif t < 256:                 counts["NOTE_OFF"] += 1
        elif t < 356:
            counts["TIME_SHIFT"] += 1
            total_ms += (t - 256 + 1) * 10
        elif t < 388:                 counts["VELOCITY"] += 1
        else:                         counts["PAD"] += 1
    n = len(token_seq)
    print(f"\nToken distribution ({n} total):")
    for k, v in counts.items():
        print(f"  {k:12s}: {v:5d}  ({v/n*100:.1f}%)")
    print(f"  Total duration : {total_ms/1000:.1f}s")
    print(f"  Music density  : {counts['NOTE_ON']} note-ons over {total_ms/1000:.1f}s\n")



# ── Token → MIDI ──────────────────────────────────────────────────────────────

def tokens_to_midi(token_seq: list, output_path: str,
                   default_tempo: int = 500_000):
    mid = mido.MidiFile(ticks_per_beat=480)
    track = mido.MidiTrack()
    mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=default_tempo, time=0))

    ticks_per_beat = 480
    ms_per_tick = default_tempo / 1_000 / ticks_per_beat

    current_velocity = 64
    pending_ticks = 0.0
    open_notes = set()          

    def flush_time():
        nonlocal pending_ticks
        t = int(round(pending_ticks))
        pending_ticks = 0.0
        return t

    for token in token_seq:
        if token == PAD_TOKEN:
            continue
        if NOTE_ON_OFFSET <= token < NOTE_ON_OFFSET + 128:
            pitch = token - NOTE_ON_OFFSET
            if pitch in open_notes:
                track.append(mido.Message(
                    "note_off", note=pitch, velocity=0, time=flush_time()
                ))
            track.append(mido.Message(
                "note_on", note=pitch,
                velocity=current_velocity, time=flush_time()
            ))
            open_notes.add(pitch)

        elif NOTE_OFF_OFFSET <= token < NOTE_OFF_OFFSET + 128:
            pitch = token - NOTE_OFF_OFFSET
            track.append(mido.Message(
                "note_off", note=pitch, velocity=0, time=flush_time()
            ))
            open_notes.discard(pitch)

        elif TIME_SHIFT_OFFSET <= token < TIME_SHIFT_OFFSET + 100:
            steps = token - TIME_SHIFT_OFFSET + 1
            pending_ticks += steps * TIME_STEP_MS / ms_per_tick

        elif VELOCITY_OFFSET <= token < VELOCITY_OFFSET + N_VELOCITY_BINS:
            bin_idx = token - VELOCITY_OFFSET
            current_velocity = bin_idx * VELOCITY_BIN_SIZE + VELOCITY_BIN_SIZE // 2

    for pitch in sorted(open_notes):
        track.append(mido.Message("note_off", note=pitch, velocity=0, time=0))

    mid.save(output_path)
    print(f"MIDI saved → {output_path}  (closed {len(open_notes)} dangling notes)")

# ── Seed extraction ────────────────────────────────────────────────────────────

def get_seed_tokens(seed_length: int = SEED_LENGTH) -> list:
    """
    Sample a random window from the test set and use its first
    seed_length non-PAD tokens as the generation prompt.
    """
    rng  = np.random.default_rng(SEED)
    data = np.load(os.path.join(PROCESSED_DIR, "test_task1.npy"))
    idx  = rng.integers(0, len(data))
    window = data[idx].tolist()
    # Strip trailing PADs, then take first seed_length tokens
    tokens = [t for t in window if t != PAD_TOKEN]
    seed   = tokens[:seed_length]
    print(f"Seed: window #{idx}, first {len(seed)} tokens")
    return seed


# ── Main ──────────────────────────────────────────────────────────────────────

def generate(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model
    model = MusicLSTM().to(device)
    ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"No checkpoint found at {ckpt_path}. Run train.py first."
        )
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
          f"(val_loss={ckpt['val_loss']:.4f})")

    # Get seed
    seed_tokens = get_seed_tokens(args.seed_length)

    # Generate
    print(f"Generating {args.tokens} tokens  "
          f"(temperature={args.temperature}, top_k={args.top_k})…")
    generated = model.generate(
        seed_tokens=seed_tokens,
        max_new_tokens=args.tokens,
        temperature=args.temperature,
        top_k=args.top_k,
        device=device,
    )
    print(f"Total tokens (seed + generated): {len(generated)}")
    diagnose_tokens(generated)

    # Decode to MIDI
    out_path = os.path.join(OUTPUT_DIR, "symbolic_unconditioned.mid")
    tokens_to_midi(generated, out_path)

    return generated


args = Args()
args.tokens = 1024 
args.temperature = 1.0
args.top_k = 20
args.seed_length = SEED_LENGTH

generate(args)


Loaded checkpoint from epoch 28 (val_loss=2.0803)
Seed: window #1393, first 64 tokens
Generating 1024 tokens  (temperature=1.0, top_k=20)…
Total tokens (seed + generated): 1088

Token distribution (1088 total):
  NOTE_ON     :   194  (17.8%)
  NOTE_OFF    :   195  (17.9%)
  TIME_SHIFT  :   525  (48.3%)
  VELOCITY    :   174  (16.0%)
  PAD         :     0  (0.0%)
  Total duration : 19.2s
  Music density  : 194 note-ons over 19.2s

MIDI saved → /home/yax025/CSE253/Assignment2/outputs/symbolic_unconditioned.mid  (closed 1 dangling notes)


[256,
 200,
 257,
 256,
 372,
 60,
 256,
 373,
 63,
 256,
 203,
 258,
 256,
 191,
 274,
 375,
 75,
 256,
 373,
 72,
 257,
 256,
 188,
 256,
 200,
 257,
 203,
 270,
 374,
 58,
 256,
 256,
 376,
 63,
 261,
 191,
 273,
 76,
 256,
 374,
 72,
 258,
 200,
 257,
 204,
 256,
 186,
 271,
 256,
 375,
 56,
 280,
 256,
 65,
 259,
 184,
 257,
 256,
 193,
 256,
 374,
 77,
 256,
 373,
 68,
 258,
 205,
 256,
 196,
 267,
 56,
 256,
 68,
 259,
 256,
 184,
 258,
 256,
 196,
 267,
 372,
 63,
 256,
 375,
 55,
 273,
 191,
 256,
 373,
 67,
 256,
 373,
 70,
 257,
 183,
 259,
 198,
 256,
 195,
 269,
 256,
 377,
 60,
 265,
 188,
 261,
 256,
 376,
 67,
 256,
 256,
 372,
 70,
 258,
 195,
 259,
 198,
 268,
 256,
 374,
 63,
 264,
 191,
 269,
 373,
 58,
 256,
 374,
 67,
 258,
 256,
 186,
 257,
 195,
 273,
 256,
 60,
 273,
 188,
 256,
 256,
 373,
 56,
 256,
 370,
 68,
 263,
 256,
 196,
 258,
 184,
 263,
 368,
 53,
 257,
 372,
 67,
 259,
 256,
 195,
 269,
 373,
 58,
 256,
 181,
 256,
 373,
 67,
 260,
 195,
 257,
 256,

In [37]:
cd ..

/home/yax025/CSE253/Assignment2


/home/yax025/CSE253/Assignment2/checkpoints/checkpoints
